In [1]:
import tensorflow as tf
import keras
import numpy as np
import cv2
import mediapipe as mp

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("NumPy:", np.__version__)
print("OpenCV:", cv2.__version__)
print("MediaPipe:", mp.__version__)

TensorFlow: 2.20.0
Keras: 3.14.0
NumPy: 1.26.4
OpenCV: 4.11.0
MediaPipe: 0.10.35


In [1]:


import cv2
import numpy as np
import mediapipe as mp
from tensorflow.keras.models import load_model

# -------------------- Load Model --------------------
model = load_model("gesture_predict_model.h5", compile=False)

# -------------------- Class Labels --------------------
class_labels = [
    'Bathroom', 'Call', 'Drink', 'Eat', 'Help',
    'Listen Up', 'No', 'Pain', 'Stop',
    'When', 'Where', 'Yes'
]

# -------------------- NEW MEDIAPIPE HAND API --------------------
BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

base_options = BaseOptions(
    model_asset_path=r"C:\Users\Rimpa\OneDrive\Desktop\SignPrediction\hand_landmarker.task"
)

options = HandLandmarkerOptions(
    base_options=base_options,
    running_mode=VisionRunningMode.VIDEO,
    num_hands=2
)

hand_detector = HandLandmarker.create_from_options(options)

# -------------------- FACE DETECTION (OpenCV Haar Cascade) --------------------
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

# -------------------- Camera --------------------
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Camera not opened")
    raise SystemExit

print("Press S to save | Press Q to quit")

while True:

    ret, frame = cap.read()
    if not ret:
        break

    h, w, _ = frame.shape

    # -------------------- RGB & Gray --------------------
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    bw_frame = cv2.cvtColor(rgb_frame, cv2.COLOR_RGB2GRAY)

    # -------------------- FACE DETECTION --------------------
    faces = face_cascade.detectMultiScale(
        bw_frame,
        scaleFactor=1.1,
        minNeighbors=5
    )
    num_faces = len(faces)

    # -------------------- Mediapipe Image --------------------
    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_frame
    )

    # -------------------- Hand Detection --------------------
    detection_result = hand_detector.detect_for_video(
        mp_image,
        int(cap.get(cv2.CAP_PROP_POS_MSEC))
    )

    num_hands = len(detection_result.hand_landmarks) if detection_result.hand_landmarks else 0

    # -------------------- Mode --------------------
    mode = "No Detection"
    if num_hands == 2:
        mode = "Two Hands"
    elif num_hands == 1 and num_faces >= 1:
        mode = "Face + One Hand"
    elif num_hands == 1:
        mode = "One Hand"

    # -------------------- ROI Mask  --------------------
    mask = np.zeros_like(bw_frame)
    all_x, all_y = [], []

    # Hand landmarks → mask
    if detection_result.hand_landmarks:
        for hand_landmarks in detection_result.hand_landmarks:
            points = np.array([
                (int(lm.x * w), int(lm.y * h))
                for lm in hand_landmarks
            ])
            cv2.fillPoly(mask, [points], 255)
            all_x.extend(points[:, 0])
            all_y.extend(points[:, 1])

    # Face → mask 
    if num_faces >= 1:
        fx, fy, fw, fh = faces[0]
        fx1, fy1 = fx, fy
        fx2, fy2 = fx + fw, fy + fh
        cv2.rectangle(mask, (fx1, fy1), (fx2, fy2), 255, -1)
        all_x.extend([fx1, fx2])
        all_y.extend([fy1, fy2])

    # -------------------- Masked Frame --------------------
    masked_frame = cv2.bitwise_and(bw_frame, mask)

    # -------------------- ROI --------------------
    roi = None

    if all_x and all_y:
        x1 = max(min(all_x), 0)
        x2 = min(max(all_x), w)
        y1 = max(min(all_y), 0)
        y2 = min(max(all_y), h)

        if x2 > x1 and y2 > y1:
            temp_roi = masked_frame[y1:y2, x1:x2]
            if temp_roi.size > 0:
                roi = cv2.resize(temp_roi, (224, 224))

    # -------------------- Prediction --------------------
    prediction = "No Prediction"

    if roi is not None and roi.size > 0:
        img = cv2.cvtColor(roi, cv2.COLOR_GRAY2RGB)
        img = img / 255.0
        img = np.reshape(img, (1, 224, 224, 3))

        pred = model.predict(img, verbose=0)
        class_index = np.argmax(pred)
        prediction = class_labels[class_index]

    # -------------------- Display --------------------
    cv2.putText(frame, f"Mode: {mode}", (30, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.putText(frame, f"Gesture: {prediction}", (30, 100),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

    cv2.imshow("Live Camera", frame)

    if roi is not None and roi.size > 0:
        cv2.imshow("ROI", roi)

    key = cv2.waitKey(1) & 0xFF

    # -------------------- Save --------------------
    if key == ord('s') and roi is not None:
        filename = f"{prediction.replace(' ', '_')}.png"
        cv2.imwrite(filename, roi)
        print(f"Saved: {filename}")

    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Press S to save | Press Q to quit


KeyboardInterrupt: 